<a href="https://colab.research.google.com/github/jonathancdelc/jonathancdelc/blob/main/Practica_5_Data_Science_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import sqlite3

# =========================================================================
# PASO 1: Creación de la Base de Datos con Datos Sensibles
# =========================================================================
def inicializar_base_de_datos():
    conexion = sqlite3.connect(":memory:") # Usamos memoria para la práctica
    cursor = conexion.cursor()

    # Tabla con datos financieros ficticios y sensibles
    cursor.execute("""
        CREATE TABLE clientes_banca (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            email TEXT,
            saldo_usd REAL,
            score_crediticio INTEGER
        )
    """)

    # Insertamos algunos datos de prueba
    datos = [
        ('Elena Pozo', 'elena@email.com', 45000.50, 780),
        ('Marcos Reus', 'marcos@email.com', 1200.00, 620),
        ('Julia Silva', 'julia@email.com', 89000.00, 810)
    ]
    cursor.executemany("INSERT INTO clientes_banca (nombre, email, saldo_usd, score_crediticio) VALUES (?, ?, ?, ?)", datos)
    conexion.commit()
    return conexion

# =========================================================================
# PASO 2 y 4: Definición de Roles e Implementación de Controles en Python
# =========================================================================
# Definimos los niveles de acceso permitidos por rol (Simulación de DCL)
ROLES_PERMISOS = {
    "DATA_SCIENTIST": ["SELECT"],                  # Solo lectura
    "DATA_ENGINEER":  ["SELECT", "INSERT", "UPDATE"], # Lectura y Escritura
    "MARKETING":      []                           # Sin acceso a esta tabla
}

def ejecutar_consulta_segura(conexion, rol_usuario, query, parametros=()):
    """
    Simula una capa intermedia de gobernanza (Proxy/DCL) que valida
    los permisos del rol antes de tocar la base de datos.
    """
    # Identificar la operación (la primera palabra de la consulta)
    operacion = query.strip().split()[0].upper()

    # Validar si el rol existe
    if rol_usuario not in ROLES_PERMISOS:
        print(f"❌ Acceso Denegado: El rol '{rol_usuario}' no está registrado en el sistema.")
        return None

    # Validar si el rol tiene permiso para esa operación específica
    if operacion not in ROLES_PERMISOS[rol_usuario]:
        print(f"⛔ Seguridad DCL: El rol '{rol_usuario}' NO tiene permiso para realizar un '{operacion}'.")
        return None

    # Si pasa el control de acceso, se ejecuta en la BD
    try:
        cursor = conexion.cursor()
        cursor.execute(query, parametros)

        if operacion == "SELECT":
            return cursor.fetchall()
        else:
            conexion.commit()
            print(f"✅ Operación '{operacion}' realizada con éxito por el rol '{rol_usuario}'.")
            return True
    except Exception as e:
        print(f"⚠️ Error de ejecución: {e}")
        return None

# =========================================================================
# PASO 3: Simulación de diferentes niveles de acceso
# =========================================================================
if __name__ == "__main__":
    # Inicializamos nuestro entorno seguro
    conn = inicializar_base_de_datos()

    print("--- SIMULACIÓN DE CONTROL DE ACCESO (DCL) ---")

    # 1. Caso: Rol Sin Acceso (Marketing intentando husmear)
    print("\n[Escenario 1: Equipo de Marketing intentando leer datos financieros]")
    query_ver = "SELECT * FROM clientes_banca"
    ejecutar_consulta_segura(conn, "MARKETING", query_ver)

    # 2. Caso: Rol de Solo Lectura (Data Scientist analizando el Score para un modelo de ML)
    print("\n[Escenario 2: Data Scientist leyendo datos para entrenar modelo]")
    datos_ml = ejecutar_consulta_segura(conn, "DATA_SCIENTIST", query_ver)
    if datos_ml:
        print("Resultados obtenidos para el modelo:")
        for fila in datos_ml:
            print(f"-> Cliente: {fila[1]} | Score: {fila[4]}")

    # 3. Caso: Rol de Solo Lectura intentando Alterar Datos (Data Scientist intentando dar un "bono")
    print("\n[Escenario 3: Data Scientist intenta modificar un saldo (Inyección/Modificación no autorizada)]")
    query_hack = "UPDATE clientes_banca SET saldo_usd = 999999 WHERE nombre = 'Marcos Reus'"
    ejecutar_consulta_segura(conn, "DATA_SCIENTIST", query_hack)

    # 4. Caso: Rol de Escritura (Data Engineer actualizando el pipeline)
    print("\n[Escenario 4: Data Engineer actualizando registros legítimamente]")
    query_update = "UPDATE clientes_banca SET saldo_usd = 1300.00 WHERE nombre = 'Marcos Reus'"
    ejecutar_consulta_segura(conn, "DATA_ENGINEER", query_update)

    # Cerramos la conexión
    conn.close()

--- SIMULACIÓN DE CONTROL DE ACCESO (DCL) ---

[Escenario 1: Equipo de Marketing intentando leer datos financieros]
⛔ Seguridad DCL: El rol 'MARKETING' NO tiene permiso para realizar un 'SELECT'.

[Escenario 2: Data Scientist leyendo datos para entrenar modelo]
Resultados obtenidos para el modelo:
-> Cliente: Elena Pozo | Score: 780
-> Cliente: Marcos Reus | Score: 620
-> Cliente: Julia Silva | Score: 810

[Escenario 3: Data Scientist intenta modificar un saldo (Inyección/Modificación no autorizada)]
⛔ Seguridad DCL: El rol 'DATA_SCIENTIST' NO tiene permiso para realizar un 'UPDATE'.

[Escenario 4: Data Engineer actualizando registros legítimamente]
✅ Operación 'UPDATE' realizada con éxito por el rol 'DATA_ENGINEER'.


# Documentación de Riesgos y Mitigación (Gobernanza de Datos)
Al diseñar este pipeline y simular las restricciones, podemos identificar los siguientes riesgos del entorno real y cómo los mitigamos conceptualmente:
1. Riesgo de Exposición: Principio del Menor Privilegio Violado
El Riesgo: Si el equipo de Data Science usa la misma cuenta de administrador de la base de datos que el equipo de Ingeniería, un error en un script de Python (o una libreta de Jupyter) podría borrar o corromper tablas enteras de producción (DROP TABLE).
Mitigación en la práctica: Se aplicó un enfoque donde el rol DATA_SCIENTIST solo tiene permitido el comando SELECT. Si intentan un UPDATE o DELETE, el sistema bloquea la petición antes de que toque el almacenamiento.
2. Riesgo de Privacidad: Datos Identificables (PII) expuestos
El Riesgo: Para entrenar un modelo de score crediticio, el modelo necesita patrones numéricos (como el saldo y el score), pero no necesita el nombre ni el correo electrónico del cliente. Dejar esos datos a la vista viola normativas como GDPR o regulaciones locales de protección de datos.
Mitigación para el futuro: Aunque en este script controlamos quién entra, una buena práctica de gobernanza adicional sería aplicar Vistas (Views) o Enmascaramiento de datos, permitiendo que el Data Scientist haga SELECT solo a una vista que excluya las columnas nombre y email.
3. Riesgo Técnico: Limitación del motor local (SQLite)
El Riesgo: SQLite no tiene control de usuarios. Si el archivo .db se guarda en un servidor compartido sin seguridad, cualquiera con acceso al sistema de archivos puede copiarlo y robar la información.
Mitigación: En entornos de producción, la solución es migrar hacia sistemas de nivel empresarial (PostgreSQL, Snowflake, BigQuery) donde el DCL se ejecuta de forma nativa mediante comandos reales:

```
GRANT SELECT ON clientes_banca TO rol_data_scientist;
REVOKE UPDATE, DELETE ON clientes_banca FROM rol_data_scientist;
```

